In [ ]:
import pandas as pd
import json

  Example of the json record we're reading
  
  ```json
  "956d8f9f18208168bb30bbac9299bb59": {
    "aiResults": [
      {
        "modelName": "speciesnet/PyTorch/v4.0.1a",
        "runDate": "2025-05-24",
        "confBlank": 0.99,
        "confHuman": 0.0,
        "confAnimal": 0.01
      }
    ]
  },
  ```

In [ ]:
with open('../mongodb_formatted_detections.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)
bins = pd.interval_range(start=0, end=1, freq=0.05, closed='left')

records = []
for key, value in data.items():
    for result in value.get('aiResults', []):
        record = {'id': key}
        record.update(result)
        records.append(record)

df = pd.DataFrame(records)
df.head()

In [ ]:

df['conf_bin'] = pd.cut(df['confAnimal'], bins)
counts = df['conf_bin'].value_counts().sort_index()


In [9]:

# When printing, round the bin edges to 2 decimals for display
rounded_counts = counts.copy()
rounded_counts.index = pd.IntervalIndex.from_tuples(
    [(round(i.left, 2), round(i.right, 2)) for i in counts.index],
    closed='left'
)
display(rounded_counts)

[0.0, 0.05)    72871
[0.05, 0.1)     6281
[0.1, 0.15)     3063
[0.15, 0.2)     1225
[0.2, 0.25)     1147
[0.25, 0.3)     1075
[0.3, 0.35)      774
[0.35, 0.4)      532
[0.4, 0.45)      609
[0.45, 0.5)      558
[0.5, 0.55)      601
[0.55, 0.6)      668
[0.6, 0.65)      549
[0.65, 0.7)     1024
[0.7, 0.75)      918
[0.75, 0.8)     1679
[0.8, 0.85)     3183
[0.85, 0.9)     2981
[0.9, 0.95)     4734
[0.95, 1.0)      465
Name: count, dtype: int64

In [ ]:
import ipywidgets as widgets

from IPython.display import display

def show_counts(threshold):
    above = (counts.index.right > threshold).sum()
    below = (counts.index.right <= threshold).sum()
    print(f"Threshold: {threshold:.2f}")
    print(f"Bins with upper edge > threshold: {above}")
    print(f"Bins with upper edge <= threshold: {below}")

slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Threshold:',
    continuous_update=False
)

widgets.interact(show_counts, threshold=slider)